# C2 · Plotting rasters: an honest map of the Kathmandu Valley DEM

> **Time:** about 2 hours  
> **You need:** Lesson C1 (opening the DEM, the transform, `rowcol`), A10 · Plotting with Matplotlib, and the vector maps of Part B (especially B11 · Making a beautiful map).  
> **By the end you can:**
> 1. draw a raster with `imshow` in map coordinates (`extent=`) and true proportions (`aspect`), with a colourbar in metres,
> 2. choose a colour map for elevation, read a histogram, and stretch the colours with percentiles,
> 3. draw district boundaries and points on top of a raster, use `rasterio.plot.show` and rioxarray's `.plot()`, add a hillshade, and save the map.

In C1 you read the DEM as numbers. A grid of 3.5 million numbers means little until you **see** it. This lesson turns the DEM into a map that is right side up, in true proportions, with colours that tell the truth.

A raster map can mislead in quiet ways: stretched shapes, colours that suggest water where there is none, a colour scale wasted on a few extreme cells. You will make each mistake on purpose once, so that you recognise it later.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA).* The full licence notice is in `data/README.md`.

**Counting lines:** as in A10 and Part B, lines that only set up a plot (`fig, ax = plt.subplots()`, `plt.show()`) do not count toward the Q1/Q2/Q3 line counts. From step 3 on, `ax.set_aspect(aspect)` is a setup line too.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 23 of 32 · [Course home](../README.md)  
← Previous: [C1 · What a raster is: the Kathmandu Valley as a grid of numbers](C01_What_a_raster_is.ipynb) · Next: [C3 · Clip, mask and subset: cutting the DEM to the Kathmandu Valley and its districts](C03_Clip_mask_and_subset.ipynb) →

## 1. A first picture with `imshow`

A raster array is already a picture: one number per cell, in rows and columns. Matplotlib's `ax.imshow(array)` ("image show") paints each cell as a small coloured square.

It gives every value a colour from a **colour map** (colormap), from the lowest value to the highest. The default colour map is called `"viridis"`: dark purple for low values, yellow for high ones.

`imshow` puts row 0 at the **top**, like a page of text. Because row 0 of the DEM is its northern edge (C1), north comes out at the top. Good.

Below, the DEM is read exactly as in C1. **Predict:** what numbers will appear along the axes?

In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.transform import rowcol

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)
DEM_PATH = DATA / "dem_kathmandu_melamchi.tif"

with rasterio.open(DEM_PATH) as src:    # as in C1
    dem_m = src.read(1)
    dem_transform = src.transform
    dem_crs = src.crs
    bounds = src.bounds

print(dem_m.shape, dem_m.dtype, dem_m.min(), dem_m.max())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(dem_m)
plt.show()

The axes count **columns** (0 to 2,159) and **rows** (0 to 1,619), not degrees. The picture is right side up: the high ground of the north-east is yellow at the top right, and the dark river valleys wind through the middle.

As a map, this picture has four problems, which the next steps fix one by one:

1. The axes show array positions, not longitude and latitude (step 2).
2. The proportions are wrong: each cell is drawn as a square, but on the ground the cells are 27.3 m wide and 30.8 m tall (C1, step 4) (step 3).
3. Without a **colourbar**, nobody can read an elevation from a colour (step 4).
4. The colours are spread from 525 m to 3,979 m, although very few cells are that low or that high (step 5).

### Practice

**Q1 · one line:** Draw only rows 600 to 1299 and columns 0 to 1299 of `dem_m`, roughly the Kathmandu Valley, by slicing as in A8.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots()
ax.imshow(dem_m[600:1300, 0:1300])
plt.show()
```

The slice is a smaller 2-D array, 700 rows × 1,300 columns, and `imshow` draws it like any other. The flat valley floor shows as one even colour in the middle. The axes start again at 0, because the slice does not remember where it came from.
</details>

**Q2 · two lines:** Add a colourbar: store what `imshow` returns as `im`, then call `fig.colorbar(im, ax=ax)`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots()
im = ax.imshow(dem_m)
fig.colorbar(im, ax=ax)
plt.show()
```

`imshow` returns the image it drew; `fig.colorbar` reads its colours and their values and draws the key beside the map. The bar runs from about 500 to about 4,000, without a unit yet: step 4 adds one.
</details>

**Q3 · three lines:** Draw the mask of cells above 2,000 m (A8) with `cmap="gray"`, and put the percentage of those cells in the title with an f-string.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
above_2000 = dem_m > 2000
fig, ax = plt.subplots()
ax.imshow(above_2000, cmap="gray")
ax.set_title(f"Cells above 2,000 m: {above_2000.mean() * 100:.1f}%")
plt.show()
```

A True/False array is drawn as two colours: `True` (1) white, `False` (0) black. The title says 15.8%, as in C1. The white areas are the ridges, mostly in the north and around the valley rim.
</details>

## 2. Putting the picture on the map: `extent`

`imshow` knows nothing about maps. You tell it where the picture's edges lie with `extent=(left, right, bottom, top)`. The axes then show map coordinates: here longitude and latitude in degrees.

Watch the order. A raster's `bounds` are `(left, bottom, right, top)`, but `extent` wants `(left, right, bottom, top)`: both x values first, then both y values. Mixing the two is a classic silent mistake, shown below on purpose.

`rasterio.plot.plotting_extent(array, transform)` builds the extent in the right order from the transform, so you cannot mix them up.

With an extent, x is the longitude and y the latitude, as always since B1. Points and lines in the same CRS can now be drawn on top of the raster.

In [ ]:
from rasterio.plot import plotting_extent, show

extent = (bounds.left, bounds.right, bounds.bottom, bounds.top)
print(extent)

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(dem_m, extent=extent)
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
plt.show()

In [ ]:
# THE ORDER MISTAKE, ON PURPOSE: bounds are (left, bottom, right, top), not an extent
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(dem_m, extent=tuple(bounds))
plt.show()

With the right extent, the x-axis runs from 85.2° to 85.8° E and the y-axis from 27.55° to 28.0° N, the area of the DEM (C1).

The wrong extent tells Matplotlib that x runs from 85.2 to 27.55 and y from 85.8 to 28.0. No error appears, and the picture still looks like the DEM. But the axes are nonsense: x runs backwards from 85 down to 28, the y-axis runs from 86 at the bottom to 28 at the top, and the map is squeezed into a square. Always read the numbers on your axes.

### Practice

**Q1 · one line:** Print `plotting_extent(dem_m, dem_transform)` and compare it with `extent`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(plotting_extent(dem_m, dem_transform))
```

It prints the same four numbers as `extent`, in the same order: left, right, bottom, top. It works them out from the transform and the array's shape, so it also works for a cut-out piece of a raster with its own transform.
</details>

**Q2 · two lines:** Draw the DEM with `extent`, then mark Kathmandu Durbar Square (85.307° E, 27.704° N, approximate) with a red triangle: `ax.plot(85.307, 27.704, "r^")`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots()
ax.imshow(dem_m, extent=extent)
ax.plot(85.307, 27.704, "r^")
plt.show()
```

The triangle lands on the flat valley floor, left of centre. It can be drawn in degrees only because the raster's axes are now in degrees too. `"r^"` means red (`r`) triangle (`^`), a Matplotlib shorthand.
</details>

**Q3 · three lines:** Zoom in on the Kathmandu Valley box from B2 (85.2–85.55° E, 27.55–27.82° N, approximate): draw the DEM with `extent`, then set the limits with `ax.set_xlim` and `ax.set_ylim` (A10).

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots()
ax.imshow(dem_m, extent=extent)
ax.set_xlim(85.2, 85.55)
ax.set_ylim(27.55, 27.82)
plt.show()
```

Limits only change the window you look through; the whole DEM is still drawn and stored. The valley floor fills the middle of the box, ringed by higher hills. In C3 you will cut the box out of the raster itself.
</details>

## 3. True proportions: the aspect

With an extent, `imshow` draws one degree of longitude exactly as long as one degree of latitude. On the ground they are not equal.

You met the rule in B1: a degree of latitude is about 111 km everywhere, but a degree of longitude is about 111 × cos(latitude) km. At the DEM's middle latitude, about 27.8° N, a degree of longitude is only 0.885 times as long. The map above is stretched from west to east by about 13%.

The fix is `ax.set_aspect(aspect)`, where `aspect` says how many times taller one unit of y is drawn than one unit of x. For a grid in degrees use `aspect = 1 / cos(middle latitude)`.

GeoPandas did this for you in Part B whenever it drew a layer in degrees. `imshow` does not, so for rasters you set it yourself. In a projected CRS such as UTM (C4) both axes are in metres and the aspect is 1.

**Predict:** will the corrected map be wider or narrower than the stretched one?

In [ ]:
mid_lat_deg = (bounds.top + bounds.bottom) / 2
aspect = 1 / math.cos(math.radians(mid_lat_deg))
print(round(mid_lat_deg, 3), round(aspect, 3))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.imshow(dem_m, extent=extent)
ax1.set_title("Stretched: 1° of x drawn as long as 1° of y")
ax2.imshow(dem_m, extent=extent)
ax2.set_aspect(aspect)
ax2.set_title(f"True proportions: aspect = {aspect:.3f}")
plt.show()

The middle latitude is 27.775° and the aspect 1.13. The corrected map is **narrower** and taller: each degree of latitude is now drawn 1.13 times as tall as a degree of longitude, as on the ground.

This is the same fact as C1's cell sizes: 30.8 m ÷ 27.3 m ≈ 1.13. A 13% stretch can slip past the eye, but it changes every shape, every slope direction you read, and every distance you estimate from the map.

From now on, every map in this lesson sets `ax.set_aspect(aspect)`.

### Practice

**Q1 · one line:** Print the aspect rounded to 2 decimal places, then check it against C1's cell sizes: print `30.8 / 27.3` rounded the same way.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round(aspect, 2), round(30.8 / 27.3, 2))
```

Both print 1.13. The aspect is the ratio of a cell's ground height to its ground width.
</details>

**Q2 · two lines:** Work out the aspect at the DEM's top edge (`bounds.top`) and at its bottom edge (`bounds.bottom`). Print both, rounded to 4 decimal places.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round(1 / math.cos(math.radians(bounds.top)), 4))
print(round(1 / math.cos(math.radians(bounds.bottom)), 4))
```

1.1326 at the top and 1.1279 at the bottom. The difference is under 0.5%, so one aspect for the whole map is fine here. For a raster covering many degrees of latitude, a projected CRS is the better answer (C4).
</details>

**Q3 · three lines:** Draw the DEM with `extent` and true proportions, and label the axes with units.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots()
ax.imshow(dem_m, extent=extent)
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
ax.set_aspect(aspect)
plt.show()
```

Three lines do the work: the image and the two labels. The map now has the right place, the right shape and readable axes. Only the colours are left to fix.
</details>

## 4. Colour maps and the colourbar

A **colour map** turns numbers into colours. Choose it to match the data:

| Kind | Colours | Use it for | Examples |
| --- | --- | --- | --- |
| **sequential** | light to dark, or one hue to another, in one direction | amounts from low to high: elevation, slope, rainfall | `"viridis"`, `"cividis"`, `"gist_earth"`, `"terrain"` |
| **diverging** | two colours meeting at a pale middle | values above and below a middle value, such as a change | `"RdBu"`, `"BrBG"` |
| **qualitative** | unrelated colours | categories, such as land cover | `"tab10"` |

Avoid **rainbow** maps such as `"jet"`. Bright bands appear where the hue changes fast, and readers see edges that are not in the data. Many colour-blind readers cannot tell its reds from its greens. (This is the "no rainbow colours" rule of A10.)

`"terrain"` and `"gist_earth"` are made for elevation, but both **start with blue**, the colour maps use for water. On a DEM with no sea, the deepest valleys then look like lakes. You can cut the blue part off: a colour map called with numbers from 0 to 1 returns colours, and `ListedColormap` turns a list of colours into a new colour map.

Every colour map needs a **colourbar** with a unit: `fig.colorbar(im, ax=ax, label="Elevation (m)")`.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, name in zip(axes.flat, ["terrain", "gist_earth", "viridis", "jet"]):
    im = ax.imshow(dem_m, extent=extent, cmap=name)
    fig.colorbar(im, ax=ax, label="Elevation (m)")
    ax.set_title(f'cmap="{name}"')
    ax.set_aspect(aspect)
plt.show()

In [ ]:
from matplotlib.colors import ListedColormap

# "terrain" without its blue start: colours from 25% to 100% of the way along it
land_cmap = ListedColormap(plt.colormaps["terrain"](np.linspace(0.25, 1, 256)))

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(dem_m, extent=extent, cmap=land_cmap)
fig.colorbar(im, ax=ax, label="Elevation (m)")
ax.set_title("Elevation, terrain colours without the blue")
ax.set_aspect(aspect)
plt.show()

`axes.flat` walks through the four panels of the 2 × 2 grid one by one, so `zip` (B7) pairs each panel with a colour-map name.

Compare the four. `"terrain"` and `"gist_earth"` paint the deep valleys in the east and north-west blue, like rivers in flood, although every cell is dry land, at least 525 m above sea level. `"viridis"` is honest but hard to read as landscape. `"jet"` shows sharp yellow and cyan edges that do not exist on the ground.

In `land_cmap`, `np.linspace(0.25, 1, 256)` makes 256 evenly spaced positions from a quarter of the way along `"terrain"` to its end. The new map runs from green through yellow and brown to white: a natural reading of low to high land.

### Practice

**Q1 · one line:** What colour does `"terrain"` give to its lowest value? Print `plt.colormaps["terrain"](0.0)`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(plt.colormaps["terrain"](0.0))
```

It prints four numbers: about (0.2, 0.2, 0.6, 1.0). They are red, green, blue and opacity, each from 0 to 1. More blue than anything else: a dark blue. `land_cmap` starts at position 0.25 instead, a green.
</details>

**Q2 · two lines:** Draw the DEM with `land_cmap` and a **horizontal** colourbar under the map: add `orientation="horizontal"` to `fig.colorbar`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots()
im = ax.imshow(dem_m, extent=extent, cmap=land_cmap)
fig.colorbar(im, ax=ax, label="Elevation (m)", orientation="horizontal")
ax.set_aspect(aspect)
plt.show()
```

A horizontal bar suits wide maps and reports. The label with its unit, `Elevation (m)`, stays essential: without it a reader cannot tell metres from feet.
</details>

**Q3 · three lines:** Map the DEM in **feet** (1 m = 3.28084 ft) with `"cividis"`, a sequential map that colour-blind readers see well, and a colourbar labelled in feet.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
dem_ft = dem_m * 3.28084
fig, ax = plt.subplots()
im = ax.imshow(dem_ft, extent=extent, cmap="cividis")
fig.colorbar(im, ax=ax, label="Elevation (ft)")
ax.set_aspect(aspect)
plt.show()
```

The map looks the same, because colours follow the values from lowest to highest whatever the unit. Only the colourbar changes, to about 1,700 to 13,000 ft. This is why the unit on the colourbar matters so much.
</details>

## 5. Histograms and a percentile stretch

A colour map spreads its colours from a lowest value, `vmin`, to a highest, `vmax`. By default these are the array's minimum and maximum. If a few cells are extreme, most colours are spent on values that hardly any cell has.

A **histogram** shows how many cells fall in each range of values (A10). `dem_m.ravel()` flattens the 2-D grid into one long row of 3.5 million values, which `ax.hist` can count.

A **percentile** is the value below which a given percentage of the cells lie. `np.percentile(dem_m, [2, 98])` gives the 2nd and 98th percentiles. Using them as `vmin` and `vmax` is a **percentile stretch**: 96% of the cells get the full range of colours, and the 2% at each end take the end colours.

Be honest about it: add `extend="both"` to the colourbar. Its pointed ends say "values continue beyond this".

**Predict:** where will the tallest bar of the histogram be?

In [ ]:
low_m, high_m = np.percentile(dem_m, [2, 98])
print("2nd and 98th percentiles:", low_m, high_m)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(dem_m.ravel(), bins=100, color="grey")
ax.axvline(low_m, color="black", linestyle="--")
ax.axvline(high_m, color="black", linestyle="--")
ax.set_xlabel("Elevation (m)")
ax.set_ylabel("Number of cells")
ax.set_title("Elevations of the DEM; dashed lines at the 2nd and 98th percentiles")
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
im1 = ax1.imshow(dem_m, extent=extent, cmap=land_cmap)
fig.colorbar(im1, ax=ax1, label="Elevation (m)")
ax1.set_title("Full range: 525 to 3,979 m")
im2 = ax2.imshow(dem_m, extent=extent, cmap=land_cmap, vmin=low_m, vmax=high_m)
fig.colorbar(im2, ax=ax2, label="Elevation (m)", extend="both")
ax2.set_title("2-98% stretch")
for ax in (ax1, ax2):
    ax.set_aspect(aspect)
plt.show()

The 2nd percentile is 700 m and the 98th is 3,011 m. Only 2% of the cells lie below 700 m, and 2% above 3,011 m.

The tallest bars stand around 1,300 to 1,400 m: the flat floor of the Kathmandu Valley, which covers a large share of the DEM. A long thin tail runs up to nearly 4,000 m.

In the stretched map the hills between 1,500 and 2,500 m show more shades, and the summits in the north-east turn white sooner. The stretch changes only the colours, never the data. Say in a caption when you use one.

### Practice

**Q1 · one line:** What are the 5th and 95th percentiles of the DEM?

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
np.percentile(dem_m, [5, 95])
```

817 m and 2,533 m. 90% of the cells lie between these two elevations.
</details>

**Q2 · two lines:** What percentage of the cells lie below 1,000 m? Use a mask and its mean, as in C1.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
below_1000 = dem_m < 1000
print(round(below_1000.mean() * 100, 1))
```

13.6% of the cells: the river valleys east and north-west of the Kathmandu Valley. The histogram shows them as the bars left of 1,000 m.
</details>

**Q3 · three lines:** Draw the DEM with a 5–95% stretch: work out the two percentiles, use them as `vmin` and `vmax`, and add a colourbar with `extend="both"`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
p5_m, p95_m = np.percentile(dem_m, [5, 95])
fig, ax = plt.subplots()
im = ax.imshow(dem_m, extent=extent, cmap=land_cmap, vmin=p5_m, vmax=p95_m)
fig.colorbar(im, ax=ax, label="Elevation (m)", extend="both")
ax.set_aspect(aspect)
plt.show()
```

A stronger stretch: 10% of the cells now take an end colour, and the middle elevations get even more shades. Stretch too hard and the high and low ground become flat patches of one colour. For most DEMs, 2–98% is a good start.
</details>

## 6. Vectors on top of a raster

A raster and vector layers can share one map if they use the **same CRS**. Check with `==` before drawing. The DEM and the Part B layers are all in EPSG:4326.

Draw in layers, from the bottom up (B11): the raster first, then polygon outlines, then points, then labels. `districts.boundary` turns the district polygons into their outlines, so the raster shows through.

Two things change when GeoPandas draws on a raster's axes. Each layer widens the axis limits to fit its own data, and each sets its own aspect. So set the limits and the aspect **after** the last layer.

Labels come from `ax.annotate` (B11). `xytext=(4, 4)` with `textcoords="offset points"` moves each label a little up and to the right of its point.

In [ ]:
districts = gpd.read_file(DATA / "nepal_districts.geojson")
hq = gpd.read_file(DATA / "district_headquarters.geojson")
print(districts.crs == dem_crs, hq.crs == dem_crs)

hq_in_dem = hq.cx[bounds.left:bounds.right, bounds.bottom:bounds.top]   # as in C1
print(hq_in_dem["name"].tolist())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))
im = ax.imshow(dem_m, extent=extent, cmap=land_cmap, vmin=low_m, vmax=high_m)
districts.boundary.plot(ax=ax, color="black", linewidth=0.8)
hq_in_dem.plot(ax=ax, color="red", markersize=25)
for name, point in zip(hq_in_dem["name"], hq_in_dem.geometry):
    ax.annotate(name, xy=(point.x, point.y), xytext=(4, 4), textcoords="offset points")

fig.colorbar(im, ax=ax, label="Elevation (m)", extend="both", shrink=0.8)
ax.set_xlim(bounds.left, bounds.right)
ax.set_ylim(bounds.bottom, bounds.top)
ax.set_aspect(aspect)
ax.set_title("District boundaries and headquarters on the DEM (2-98% stretch)")
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
plt.show()

Both CRS checks print `True`, and five headquarters lie inside the DEM, as in C1.

The black lines are district borders. The Kathmandu Valley, with Kathmandu, Lalitpur and Bhaktapur, sits on the green floor in the south-west. Chautara, the headquarters of Sindhupalchok, lies in the east, and Dhulikhel in the south, east of the valley.

Without the `set_xlim` and `set_ylim` lines, the map would zoom out to all of Nepal, because the districts layer covers the whole country.

*Map caption: Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA). District boundaries: geoJSON-Nepal (MIT licence).*

### Practice

**Q1 · one line:** How many districts touch the DEM's box? Select them with `.cx` and the DEM's `bounds`, then count them with `len`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
len(districts.cx[bounds.left:bounds.right, bounds.bottom:bounds.top])
```

9 districts: Bhaktapur, Dhading, Kabhrepalanchok, Kathmandu, Lalitpur, Makawanpur, Nuwakot, Rasuwa and Sindhupalchok. Some only touch the edge with a sliver; C3 measures how much of each lies inside.
</details>

**Q2 · two lines:** Draw the DEM, and on top only the outline of Kathmandu district, in black. Select the district by name, as in B5.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots()
ax.imshow(dem_m, extent=extent, cmap=land_cmap)
districts[districts["district"] == "Kathmandu"].boundary.plot(ax=ax, color="black")
ax.set_aspect(aspect)
plt.show()
```

Kathmandu district lies almost entirely inside the DEM. A thin sliver crosses the western edge, so GeoPandas widens the axes a little and leaves a white strip on the left. The district's northern border follows the hills above the valley; in the south it cuts across the flat valley floor, where it meets Lalitpur and Bhaktapur.
</details>

**Q3 · three lines:** Add the made-up practice landslides: read `practice_landslides_melamchi.csv`, make a GeoDataFrame from its `lon` and `lat` columns (B3) in EPSG:4326, and draw it on the DEM as small black dots.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
slides = pd.read_csv(DATA / "practice_landslides_melamchi.csv")
slides_gdf = gpd.GeoDataFrame(slides, geometry=gpd.points_from_xy(slides["lon"], slides["lat"]), crs="EPSG:4326")
fig, ax = plt.subplots()
ax.imshow(dem_m, extent=extent, cmap=land_cmap)
slides_gdf.plot(ax=ax, color="black", markersize=8)
ax.set_aspect(aspect)
plt.show()
```

The 60 points are **made-up practice data**, placed more often on steep slopes. They gather on valley sides, not on the flat valley floor. The GeoDataFrame needs `crs="EPSG:4326"` to match the DEM; C7 reads the elevation and slope under each point.
</details>

## 7. Shortcuts: `rasterio.plot.show` and rioxarray

Two libraries can place a raster for you.

`rasterio.plot.show(src, ax=ax)` reads band 1 of an open file and draws it with the right extent, taken from the transform. It does not correct the aspect or add a colourbar.

**rioxarray** opens a raster as an xarray **DataArray**: a NumPy array that also carries labels. Each column has an **x coordinate** and each row a **y coordinate**: the longitude and latitude of the cell centres. Many later lessons use it.

```python
import rioxarray as rxr
dem_da = rxr.open_rasterio(DEM_PATH, masked=True).squeeze("band", drop=True)
```

- `masked=True` turns nodata cells into `NaN` ("not a number"), which calculations skip. The values become decimals (`float32`).
- The file has one band, so the DataArray has a `band` dimension of length 1. `.squeeze("band", drop=True)` removes it, leaving a 2-D grid of y and x.
- `dem_da.rio` holds the raster facts: `.rio.crs`, `.rio.bounds()`, `.rio.resolution()`.
- `dem_da.plot.imshow(ax=ax)` draws it with labelled axes and a colourbar.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
with rasterio.open(DEM_PATH) as src:
    show(src, ax=ax, cmap=land_cmap)
ax.set_aspect(aspect)
ax.set_title("rasterio.plot.show: extent from the transform")
plt.show()

In [ ]:
import rioxarray as rxr

dem_da = rxr.open_rasterio(DEM_PATH, masked=True).squeeze("band", drop=True)
print(dem_da.dims, dem_da.shape, dem_da.dtype)
print(dem_da.rio.crs, dem_da.rio.resolution())
dem_da

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
dem_da.plot.imshow(ax=ax, cmap=land_cmap, cbar_kwargs={"label": "Elevation (m)"})
ax.set_aspect(aspect)
ax.set_title("rioxarray: dem_da.plot.imshow()")
plt.show()

`show` placed the DEM in degrees with no extent from you. The aspect line is still yours to add.

The DataArray has the dimensions `('y', 'x')`, 1,620 × 2,160 values of type `float32`, and the same CRS and cell size as in C1. The y resolution is negative, like the transform's `e`. Its display lists the coordinates: y runs from 28.0 down to 27.55, x from 85.2 to 85.8.

`.plot.imshow()` labelled the axes `x` and `y` from the coordinate names, and `cbar_kwargs` passed the label to the colourbar. `dem_da.plot()` also works, but draws each cell as a separate shape, which is slower for large rasters.

### Practice

**Q1 · one line:** Print the bounds of `dem_da` with `.rio.bounds()`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(dem_da.rio.bounds())
```

(85.1998…, 27.5501…, 85.7998…, 28.0001…): the same box as rasterio's `bounds`, in the same order: left, bottom, right, top.
</details>

**Q2 · two lines:** A DataArray can be indexed by **coordinates** instead of row and column. Find the elevation at Kathmandu Durbar Square with `dem_da.sel(x=85.307, y=27.704, method="nearest")`, and print it as a number with `float()`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
durbar = dem_da.sel(x=85.307, y=27.704, method="nearest")
print(float(durbar))
```

1300.0 m, the same cell as `src.index` found in C1. `method="nearest"` picks the cell centre closest to the point. Unlike `rowcol`, you never touch row and column numbers.
</details>

**Q3 · three lines:** The x coordinates are cell **centres**. Print the first x coordinate, `dem_da.x.values[0]`, and `bounds.left`, then print their difference in cells (1 cell = 1/3600°).

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(dem_da.x.values[0])
print(bounds.left)
print((dem_da.x.values[0] - bounds.left) * 3600)
```

85.2 and 85.19986…: the first centre lies half a cell (0.5) inside the left edge. This is the half-cell offset you met in C1. `extent` uses the edges; coordinates use the centres.
</details>

## 8. A quick hillshade, and saving the map

A **hillshade** shows the terrain as if lit by a low sun. Slopes facing the sun are bright and slopes facing away are dark. Maps light from the north-west: **azimuth** 315° (the compass direction of the sun) and **altitude** 45° (its height above the horizon).

Lesson C6 explains the maths and computes it yourself. Here, Matplotlib's `LightSource` does it in one line. It needs the cell size in the **same unit as the elevations**: metres. Use C1's cell sizes, `dx=27.3` m (west–east) and `dy=30.8` m (south–north). Giving it degrees, or nothing, would make the hills absurdly steep.

A classic map style draws the hillshade in grey, then the elevation colours on top with `alpha=0.4` (40% opaque), so the relief shows through.

Finish by saving the map (A10): `fig.savefig(path, dpi=200, bbox_inches="tight")`. Put the data credit **on** the figure with `fig.text`, so it travels with the image.

In [ ]:
from matplotlib.colors import LightSource

sun = LightSource(azdeg=315, altdeg=45)
hillshade = sun.hillshade(dem_m, dx=27.3, dy=30.8)   # cell sizes in metres, from C1
print(hillshade.shape, hillshade.dtype, hillshade.min(), hillshade.max())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))
ax.imshow(hillshade, extent=extent, cmap="gray")
im = ax.imshow(dem_m, extent=extent, cmap=land_cmap, vmin=low_m, vmax=high_m, alpha=0.4)
districts.boundary.plot(ax=ax, color="black", linewidth=0.8)
hq_in_dem.plot(ax=ax, color="red", markersize=25)
for name, point in zip(hq_in_dem["name"], hq_in_dem.geometry):
    ax.annotate(name, xy=(point.x, point.y), xytext=(4, 4), textcoords="offset points")

fig.colorbar(im, ax=ax, label="Elevation (m)", extend="both", shrink=0.8)
ax.set_xlim(bounds.left, bounds.right)
ax.set_ylim(bounds.bottom, bounds.top)
ax.set_aspect(aspect)
ax.set_title("Kathmandu Valley and Melamchi area: elevation over hillshade")
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
fig.text(0.01, 0.01, "Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018,\n"
         "provided under COPERNICUS by the European Union and ESA). Boundaries: geoJSON-Nepal (MIT). Colours stretched 2-98%.",
         fontsize=7)

map_path = OUT / "c2_dem_hillshade_map.png"
fig.savefig(map_path, dpi=200, bbox_inches="tight")
plt.show()
print(map_path.name, round(map_path.stat().st_size / 1000), "KB")

The hillshade is a grid of the same shape as the DEM, with brightness values from 0.0 (black) to 1.0 (white).

On the map, ridges and valleys now stand out in 3-D. The deep east-side valleys and the flat Kathmandu Valley floor stand clearly apart, and the district borders follow ridges and rivers. The credit, the source of the boundaries, and the stretch are printed on the figure itself.

The file in `../outputs` takes about 2,500 KB at 200 dpi (the exact size depends on your library versions).

### Practice

**Q1 · one line:** Print the mean brightness of `hillshade`, rounded to 2 decimal places.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round(hillshade.mean(), 2))
```

0.75: most of the ground is lit fairly brightly. With the sun 45° above the horizon, only steep slopes facing away from it turn dark.
</details>

**Q2 · two lines:** Light the DEM from the **south-east** instead (azimuth 135°) and draw the hillshade in grey.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
hs_se = LightSource(azdeg=135, altdeg=45).hillshade(dem_m, dx=27.3, dy=30.8)
fig, ax = plt.subplots()
ax.imshow(hs_se, extent=extent, cmap="gray")
ax.set_aspect(aspect)
plt.show()
```

Many people now see the valleys as ridges and the ridges as valleys. Our eyes expect light from above and from the top of the page. This is why maps light from the north-west.
</details>

**Q3 · three lines:** Save a simpler map: the hillshade in grey only, with a title, as `c2_hillshade.png` at 100 dpi. Then print the file size in KB.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots()
ax.imshow(hillshade, extent=extent, cmap="gray")
ax.set_title("Hillshade, sun from the north-west")
ax.set_aspect(aspect)
fig.savefig(OUT / "c2_hillshade.png", dpi=100, bbox_inches="tight")
plt.show()
print(round((OUT / "c2_hillshade.png").stat().st_size / 1000), "KB")
```

About 200 KB, against about 2,500 KB for the 200 dpi map with colours. Fewer dots per inch make a smaller file but a less sharp picture. Use about 100 dpi for screens and 300 dpi for print (A10). This picture shows only the shading; anyone who reuses it still needs the Copernicus credit.
</details>

## 9. When it breaks: an image with the wrong shape

In C1 you saw that `src.read()` without a band number returns a 3-D array. Here it goes straight into `imshow`. Run the cell.

In [ ]:
with rasterio.open(DEM_PATH) as src:
    all_bands = src.read()

fig, ax = plt.subplots()
ax.imshow(all_bands)

The last line of the error says:

`TypeError: Invalid shape (1, 1620, 2160) for image data`

- **TypeError** means a function got something of a kind it cannot use.
- **Invalid shape (1, 1620, 2160)** names the problem: a 3-D array with 1 band first.

`imshow` accepts a 2-D array (one value per cell), or a 3-D array with the colours **last**: (rows, columns, 3) for red, green and blue. Rasterio puts the bands **first**: (bands, rows, columns). Satellite images in Part D meet this again.

The fix: read one band with `src.read(1)`, or take band 1 from the 3-D array with `all_bands[0]`.

**Try it:** fix the cell.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · A map of the Kathmandu Valley box

Cut the Kathmandu Valley box (85.2–85.55° E, 27.55–27.82° N, approximate) out of `dem_m` by slicing, and map it in the right place. Find the rows and columns of the top-left and bottom-right corners with `rowcol` (C1), slice the array, and work out the extent of the slice with `dem_transform @ (col, row)` at the two corners. Draw it with `land_cmap`, a colourbar with units and true proportions, and print the slice's shape.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
row0, col0 = rowcol(dem_transform, 85.2, 27.82)    # top-left corner
row1, col1 = rowcol(dem_transform, 85.55, 27.55)   # bottom-right corner
valley_m = dem_m[row0:row1, col0:col1]
left, top = dem_transform @ (col0, row0)
right, bottom = dem_transform @ (col1, row1)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(valley_m, extent=(left, right, bottom, top), cmap=land_cmap)
fig.colorbar(im, ax=ax, label="Elevation (m)")
ax.set_aspect(aspect)
ax.set_title("Kathmandu Valley box (approximate)")
plt.show()
print(valley_m.shape, round(left, 3), round(right, 3), round(bottom, 3), round(top, 3))
```

The slice has 972 rows and 1,260 columns, and its extent is 85.2 to 85.55° E and 27.55 to 27.82° N (rounded). The slice needs its **own** extent: drawn with the full DEM's extent, it would be stretched over the whole area. In C3, `rasterio.windows` and rioxarray's `clip_box` do this bookkeeping for you.
</details>

### Review 2 · Map and histogram side by side

Make one figure with two panels: on the left the DEM with a 2–98% stretch, on the right its histogram. Draw dashed vertical lines at `low_m` and `high_m` on the histogram, and write in its title how many cells lie outside the stretch.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
outside = (dem_m < low_m).sum() + (dem_m > high_m).sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
im = ax1.imshow(dem_m, extent=extent, cmap=land_cmap, vmin=low_m, vmax=high_m)
fig.colorbar(im, ax=ax1, label="Elevation (m)", extend="both")
ax1.set_aspect(aspect)
ax2.hist(dem_m.ravel(), bins=100, color="grey")
ax2.axvline(low_m, color="black", linestyle="--")
ax2.axvline(high_m, color="black", linestyle="--")
ax2.set_xlabel("Elevation (m)")
ax2.set_ylabel("Number of cells")
ax2.set_title(f"{outside:,} cells outside {low_m:.0f}-{high_m:.0f} m")
fig.tight_layout()
plt.show()
```

139,426 cells lie outside 700–3,011 m, about 4% of the 3.5 million. `(mask).sum()` counts `True` values, and `fig.tight_layout()` (A10) keeps the colourbar label clear of the histogram. Showing the histogram beside the map tells the reader exactly how the colours were chosen.
</details>

### Review 3 · Where is the ground above 2,500 m?

Draw the hillshade in grey. On top, show only the cells above 2,500 m in red, half transparent. Use `np.where(dem_m > 2500, 1.0, np.nan)` (A8): `imshow` leaves `NaN` cells transparent. Put the percentage of cells above 2,500 m in the title.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
high_ground = np.where(dem_m > 2500, 1.0, np.nan)
share_pct = (dem_m > 2500).mean() * 100

fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(hillshade, extent=extent, cmap="gray")
ax.imshow(high_ground, extent=extent, cmap="autumn", alpha=0.6)
ax.set_aspect(aspect)
ax.set_title(f"Ground above 2,500 m (red): {share_pct:.1f}% of the cells")
plt.show()
```

5.3% of the cells lie above 2,500 m, almost all in the north and north-east, with small patches on the valley rim. `NaN` is not a number, so `imshow` gives it no colour and the hillshade shows through. C3 uses the same trick when clipping turns outside cells into `NaN`.
</details>

### Review 4 · Your own finished map

Make and save a finished map of the DEM in `land_cmap` colours, half transparent over the hillshade, with Sindhupalchok's outline in red and all other district outlines in black. Use a 2–98% stretch, a colourbar with units, axis labels, a title, the Copernicus credit with `fig.text`, and true proportions. Save it as `c2_sindhupalchok_map.png` at 150 dpi and print whether the file exists.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
sindhu = districts[districts["district"] == "Sindhupalchok"]

fig, ax = plt.subplots(figsize=(10, 7.5))
ax.imshow(hillshade, extent=extent, cmap="gray")
im = ax.imshow(dem_m, extent=extent, cmap=land_cmap, vmin=low_m, vmax=high_m, alpha=0.5)
districts.boundary.plot(ax=ax, color="black", linewidth=0.6)
sindhu.boundary.plot(ax=ax, color="red", linewidth=2)
fig.colorbar(im, ax=ax, label="Elevation (m)", extend="both", shrink=0.8)
ax.set_xlim(bounds.left, bounds.right)
ax.set_ylim(bounds.bottom, bounds.top)
ax.set_aspect(aspect)
ax.set_title("Sindhupalchok district (red) within the DEM")
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
fig.text(0.01, 0.01, "Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, "
         "provided under COPERNICUS by the EU and ESA)", fontsize=6)
fig.savefig(OUT / "c2_sindhupalchok_map.png", dpi=150, bbox_inches="tight")
plt.show()
print((OUT / "c2_sindhupalchok_map.png").exists())
```

It prints `True`. Only part of Sindhupalchok lies inside the DEM: the red line is its western and southern border, and the district continues beyond the north and east edges of the map. The red line is drawn after the black ones, so it stays on top. Every choice on this map (colours, stretch, layer order, aspect, credit) came from one step of this lesson.
</details>

## Recap

- `ax.imshow(array, extent=(left, right, bottom, top))` draws a raster in map coordinates. `bounds` come in a different order; `plotting_extent` builds the extent for you.
- Grids in degrees need `ax.set_aspect(1 / cos(middle latitude))`, about 1.13 here, or the map is stretched west–east. Set it after the last GeoPandas layer.
- Use a **sequential** colour map for elevation, avoid rainbow maps, watch for blue that suggests water, and always add a colourbar with units.
- A **histogram** shows the spread of values; a **percentile stretch** (`vmin`, `vmax` from `np.percentile`) uses the colours where the cells are. Mark it with `extend="both"` and a caption.
- Vectors go on top of a raster when the CRS matches. `rasterio.plot.show` and rioxarray's `.plot.imshow()` place rasters for you; a **hillshade** under the colours shows the relief. Save with `fig.savefig` and put the data credit on the figure.

### Words you learned

| Word | Meaning |
| --- | --- |
| `imshow` | Matplotlib's function that draws an array as an image, one coloured square per cell |
| colour map (colormap) | A rule that turns numbers into colours |
| sequential / diverging / qualitative | Colour maps for amounts in one direction / values around a middle / categories |
| colourbar | The key beside a raster map that links colours to values, with a unit |
| extent | The map coordinates of the picture's edges: (left, right, bottom, top) |
| aspect | How many times taller one unit of y is drawn than one unit of x |
| histogram | A bar chart of how many values fall in each range |
| percentile | The value below which a given percentage of the values lie |
| percentile stretch | Using two percentiles as the lowest and highest colours (`vmin`, `vmax`) |
| `.ravel()` | Flattens a 2-D array into one long 1-D array |
| rioxarray | A library that opens rasters as xarray DataArrays, with raster tools under `.rio` |
| DataArray | An xarray array with labelled dimensions and coordinates, such as x and y |
| `masked=True` | Option of `open_rasterio` that turns nodata cells into `NaN` |
| `NaN` | "Not a number": a decimal value marking a missing cell; skipped by calculations, transparent in `imshow` |
| hillshade | A grey picture of the terrain lit by an imaginary sun |
| azimuth / altitude | The compass direction of the sun / its height above the horizon, in degrees |
| alpha | Opacity of a layer, from 0 (invisible) to 1 (solid) |

## Next

**C3 · Clip, mask and subset:** read only the part of the DEM you need, cut it to the Kathmandu Valley box and to district boundaries, and save the pieces as new files.

<!-- pygile-nav -->
**PyGILE Course** · lesson 23 of 32 · [Course home](../README.md)  
← Previous: [C1 · What a raster is: the Kathmandu Valley as a grid of numbers](C01_What_a_raster_is.ipynb) · Next: [C3 · Clip, mask and subset: cutting the DEM to the Kathmandu Valley and its districts](C03_Clip_mask_and_subset.ipynb) →